# GPy: October 2026 fixes demo

Runnable tour of user-facing improvements that landed on `devel` in October 2026.
Design write-up and credit: [CIP-0006](../cip/cip0006.md).

**Primary author of the fix PRs:** [Raashish Aggarwal](https://github.com/raashish1601) (@raashish1601).

| Area | PR | What improved |
|------|----|----------------|
| Serialization | #1124, #1127, #1128, #1132 | `WarpedGP`, `SparseGPRegression`, classification round-trips; `load_model` keeps the right class |
| Likelihoods | #1125, #1130, #1133, #1135, #1136, #1137 | Gamma+Laplace; predictive moments & LPD quadrature; Weibull samples; Bernoulli VE; MixedNoise LPD |
| Kernels | #1126 | `Symmetric` accepts runtime-built `"odd"` / `"even"` |
| Normalizer | #1138 | `posterior_samples` and `log_predictive_density` on the scale of `Y` |

Infrastructure from the same period (not demoed here): #1131 (import file handles), #1134 (PyPI deploy gating).

In [ ]:
import os
import tempfile
import warnings

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

import GPy

np.random.seed(42)
print("GPy", getattr(GPy, "__version__", "?"))
print("NumPy", np.__version__)


def as_float(x):
    return float(np.asarray(x).reshape(-1)[0])

## 1. Model serialization round-trips

`save_model` writes `<name>.json.zip` when `compress=True`; pass that full path to `load_model` (#1124, #1127, #1128, #1132).

In [ ]:
def roundtrip(model, label):
    base = os.path.join(tempfile.gettempdir(), f"gpy_demo_{label}.json")
    path = base + ".zip"
    if os.path.exists(path):
        os.remove(path)
    model.save_model(base, compress=True, save_data=True)
    loaded = type(model).load_model(path)
    os.remove(path)
    assert type(loaded) is type(model), (type(loaded), type(model))
    np.testing.assert_allclose(loaded.log_likelihood(), model.log_likelihood(), rtol=1e-5)
    print(f"{label:28s}  {type(loaded).__name__:22s}  L={as_float(loaded.log_likelihood()):.4f}  OK")
    return loaded


X = np.linspace(0, 1, 25)[:, None]
Y = np.sin(2 * np.pi * X) + 0.1 * np.random.randn(*X.shape)

m_reg = GPy.models.GPRegression(X, Y, GPy.kern.RBF(1))
m_reg.optimize(max_iters=50)
roundtrip(m_reg, "GPRegression")

m_sparse = GPy.models.SparseGPRegression(X, Y, num_inducing=5)
m_sparse.optimize(max_iters=50)
roundtrip(m_sparse, "SparseGPRegression")

m_warp = GPy.models.WarpedGP(X, Y)
m_warp.optimize(max_iters=50)
roundtrip(m_warp, "WarpedGP")

Xc = np.linspace(-2, 2, 40)[:, None]
Yc = (np.sin(Xc) > 0).astype(float)
m_cls = GPy.models.GPClassification(Xc, Yc, GPy.kern.RBF(1))
m_cls.optimize(max_iters=50)
roundtrip(m_cls, "GPClassification")

## 2. Gamma likelihood with Laplace (#1125)

Observation-space `predict` still needs `conditional_mean` on Gamma (backlog); here we show Laplace training and the latent posterior.

In [ ]:
Xg = np.linspace(0.1, 2.5, 30)[:, None]
link_f = np.exp(np.sin(Xg) + 1.0)
beta = 1.5
Yg = np.random.gamma(shape=beta * link_f, scale=1.0 / beta)

m_gamma = GPy.core.GP(
    Xg,
    Yg,
    GPy.kern.RBF(1),
    GPy.likelihoods.Gamma(beta=beta),
    inference_method=GPy.inference.latent_function_inference.Laplace(),
)
m_gamma.likelihood.beta.unfix()
m_gamma.likelihood.beta.constrain_positive()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    m_gamma.optimize(max_iters=40)

Xt = np.linspace(0.1, 2.5, 80)[:, None]
mu_f, var_f = m_gamma.predict(Xt, include_likelihood=False)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(Xg, np.log(Yg), "k.", label="log observations")
ax.plot(Xt, mu_f, "C0-", label="latent posterior mean")
ax.fill_between(
    Xt[:, 0],
    (mu_f - 2 * np.sqrt(np.maximum(var_f, 0)))[:, 0],
    (mu_f + 2 * np.sqrt(np.maximum(var_f, 0)))[:, 0],
    color="C0",
    alpha=0.2,
)
ax.set_title("Gamma + Laplace (latent GP)")
ax.legend()
plt.tight_layout()
plt.show()
print("log-likelihood:", as_float(m_gamma.log_likelihood()), "  beta:", as_float(m_gamma.likelihood.beta))

## 3. Predictive mean / variance under NumPy 2 (#1130)

Poisson Log-link moments via quadrature vs closed form.

In [ ]:
lik_pois = GPy.likelihoods.Poisson()
mu = np.array([[0.0], [1.0], [2.0], [3.5]])
var = np.array([[0.5], [0.2], [0.05], [0.01]])

mean = lik_pois.predictive_mean(mu, var)
variance = lik_pois.predictive_variance(mu, var, mean)
expected_mean = np.exp(mu + var / 2)
expected_variance = expected_mean + (np.exp(var) - 1) * np.exp(2 * mu + var)
np.testing.assert_allclose(mean, expected_mean, rtol=1e-5)
np.testing.assert_allclose(variance, expected_variance, rtol=1e-4)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(mu[:, 0], mean[:, 0], "C0o-", label="quadrature E[y]")
ax.plot(mu[:, 0], expected_mean[:, 0], "k--", label="analytic E[y]")
ax.set_xlabel("latent mean m")
ax.set_title("Poisson predictive moments (NumPy 2 safe)")
ax.legend()
plt.tight_layout()
plt.show()
print("max |mean error|", float(np.max(np.abs(mean - expected_mean))))

## 4. Weibull samples match `logpdf` (#1133)

In [ ]:
from GPy.likelihoods import Weibull

w = Weibull(beta=2.5)
f_latent = 0.0
draws = w.samples(np.full((8000, 1), f_latent))
ys = np.linspace(max(float(draws.min()), 1e-6), float(np.quantile(draws, 0.99)), 200)[:, None]
density = np.exp(w.logpdf(np.full_like(ys, f_latent), ys)[:, 0])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(draws[:, 0], bins=50, density=True, alpha=0.55, color="C1", label="samples")
ax.plot(ys[:, 0], density, "k-", lw=2, label="exp(logpdf)")
ax.set_title("Weibull samples vs logpdf")
ax.legend()
plt.tight_layout()
plt.show()
print("sample mean:", float(draws.mean()))

## 5. Symmetric kernel string identity (#1126)

Second argument is the symmetry transform matrix; third is `"odd"` or `"even"` (compared by value).

In [ ]:
transform = -np.array([[1.0]])
symmetry_type = "".join(["o", "dd"])
k_sym = GPy.kern.Symmetric(GPy.kern.RBF(1), transform, symmetry_type)
assert k_sym.symmetry_sign == -1.0
K = k_sym.K(np.linspace(-1, 1, 12)[:, None])
assert np.allclose(K, K.T)
print("symmetry_type={!r}  sign={}".format(symmetry_type, k_sym.symmetry_sign))

## 6. Bernoulli variational expectations (#1135)

Tail-stable `E[log Φ(yf)]` via `log_ndtr`, matching `scipy.stats.norm.logcdf` for tiny latent variance.

In [ ]:
lik_b = GPy.likelihoods.Bernoulli()
Y_b = np.array([[1.0], [1.0], [0.0], [0.0]])
m_b = np.array([[-8.0], [0.5], [9.0], [-1.0]])
v_b = np.full_like(m_b, 1e-8)
F, dF_dm, dF_dv, _ = lik_b.variational_expectations(Y_b, m_b, v_b)
z = np.where(Y_b == 1, 1.0, -1.0) * m_b
np.testing.assert_allclose(F, stats.norm.logcdf(z), rtol=1e-6)
print("F:     ", np.round(F.ravel(), 6))
print("logcdf:", np.round(stats.norm.logcdf(z).ravel(), 6))
print("Bernoulli VE matches logcdf in the tails: OK")

## 7. `log_predictive_density` quadrature (#1136)

Student-t LPD under a wide latent posterior (previously could return nonsense under NumPy 2 / bad limits).

In [ ]:
lik_t = GPy.likelihoods.StudentT(deg_free=4, sigma2=0.5)
y_t = np.array([[0.3], [2.5]])
mu_t = np.array([[0.2], [-0.5]])
var_t = np.array([[0.3], [24.0]])  # second point: very wide
lpd_t = lik_t.log_predictive_density(y_t, mu_t, var_t)
assert np.all(np.isfinite(lpd_t))
print("Student-t LPD:", np.round(lpd_t.ravel(), 4))

## 8. MixedNoise `log_predictive_density` (#1137)

Coregionalized regression routes each test point to the right Gaussian noise via `output_index`.

In [ ]:
X1 = np.random.rand(20, 1) * 8
X2 = np.random.rand(15, 1) * 5
Y1 = np.sin(X1) + 0.05 * np.random.randn(*X1.shape)
Y2 = -np.sin(X2) + 0.05 * np.random.randn(*X2.shape)
m_mo = GPy.models.GPCoregionalizedRegression(
    X_list=[X1, X2], Y_list=[Y1, Y2], kernel=GPy.kern.RBF(1)
)
m_mo.mixed_noise.Gaussian_noise_0.variance = 0.01
m_mo.mixed_noise.Gaussian_noise_1.variance = 0.05

X_test = np.vstack(
    [np.hstack([X1[:3], np.zeros((3, 1))]), np.hstack([X2[:3], np.ones((3, 1))])]
)
Y_test = np.vstack([Y1[:3], Y2[:3]])
Y_metadata = {"output_index": X_test[:, 1:].astype(int)}
lpd_mo = m_mo.log_predictive_density(X_test, Y_test, Y_metadata=Y_metadata)
mu_mo, var_mo = m_mo.predict(X_test, Y_metadata=Y_metadata)
expected_mo = -0.5 * np.log(2 * np.pi * var_mo) - 0.5 * (Y_test - mu_mo) ** 2 / var_mo
np.testing.assert_allclose(lpd_mo, expected_mo)
print("MixedNoise LPD matches Gaussian log density: OK")
print(np.round(lpd_mo.ravel(), 4))

## 9. Normalizer scale in samples and LPD (#1138)

With `normalizer=True`, samples and predictive density must live on the original `Y` scale (same as `predict`). See also `validate_pr1138_normalizer.py`.

In [ ]:
np.random.seed(0)
Xn = np.random.uniform(0, 10, (30, 1))
Yn = 50 + 10 * np.sin(Xn) + np.random.randn(30, 1)
X_new = np.linspace(0, 10, 4)[:, None]
Y_new = 50 + 10 * np.sin(X_new)

m_n = GPy.models.GPRegression(Xn, Yn, normalizer=True)
m_n.optimize()
mu_n, var_n = m_n.predict(X_new)
samples_n = m_n.posterior_samples(X_new, size=40000)
np.testing.assert_allclose(samples_n.mean(-1), mu_n, atol=0.1)
np.testing.assert_allclose(samples_n.var(-1), var_n, rtol=0.05)

expected_n = -0.5 * np.log(2 * np.pi * var_n) - 0.5 * (Y_new - mu_n) ** 2 / var_n
np.testing.assert_allclose(m_n.log_predictive_density(X_new, Y_new), expected_n)

print("predict var[0]:", as_float(var_n[0]))
print("sample  var[0]:", as_float(samples_n.var(-1)[0]))
print("normalizer samples + LPD: OK")

## Takeaways

- Prefer `ModelClass.load_model(path)` so the loaded object keeps that type.
- Non-Gaussian predictive moments and LPD are safe under current NumPy.
- With `normalizer=True`, trust `predict`, `posterior_samples`, and `log_predictive_density` on the original `Y` scale.
- Credit for the fix PRs: **Raashish Aggarwal (@raashish1601)** — see CIP-0006.

Follow-ups: Gamma observation-space `predict`; multi-output `posterior_samples_f` + normalizer (backlog under CIP-0006).